# Running `torchcrop` on the SIMPLACE Europe dataset

Drives the differentiable LINTUL-5 model
([`torchcrop`](https://geonextgis.github.io/torchcrop), installed editable at
`/beegfs/muduchuru/pkgs_fnl/torchcrop`) with the 10 km European SIMPLACE inputs
produced by `data4simplace` and stored under
`/data01/FDS/muduchuru/Data/SIMPLACE/europe_torchcrop`.

**Kernel:** the `sdba` conda environment
(`/home/muduchuru/miniforge3/envs/sdba`), which has `torch 2.6.0` and
`torchcrop 1.0.0`.

**Run every cell in order** (Run All). Sections 1 and 3–8 each define names the
later ones use, so a skipped cell surfaces much further down as a `NameError`
on something unrelated — skip §1.1 and §1.2 fails on `weather_ids`, which then
leaves `runnable` undefined all the way to §6. Only §2 is safe to skip.

## What is on disk

| Product | Path | Cells | Span |
| --- | --- | --- | --- |
| Weather | `weather/daily_mean_RES1_C{col}R{row}.csv.gz` | 70 705 | 1979-01-01 – 2024-12-31, daily |
| Soil | `soil/soil.csv` (+ `soil_1..3.csv` per class) | 70 705 | 6 layers to 2 m |
| Management | `management/fertilizer_winter_wheat.csv` | 68 685 | per-cell NPK schedule on a DVS axis |

Cells are keyed by `SimplaceID` on the 0.1° grid
(`lon −17…52`, `lat 34…72`, row-major from the NW corner), so
`SimplaceID → (row, col) → (lat, lon) → weather filename` is a closed-form map —
there is no cell table on disk and none is needed.

## What is *not* on disk

The gap audit in [§2](#2-data-availability-audit) prints this against the live
files. In short, six torchcrop inputs have **no** European data behind them and
are filled from constants declared in `ASSUMPTIONS`:

| torchcrop input | Status | Fill used here |
| --- | --- | --- |
| `wind` (weather ch. 7) | **export-dependent** — `Windspeed` is `-99.9` in every row of the current export. MSWX has `SFCWIND`, and `weather_export.py` now maps it (as the 2 m equivalent of the 10 m product), so a re-export fills it; the notebook uses the real column whenever it is present | constant 2 m s⁻¹ *only if absent* |
| `vp` (weather ch. 6) | **derivable** — no vapour-pressure column, but `RelHumCalc` + temperature give it | `e_s(T_mean) · RH/100` |
| `site.idpl` (sowing DOY) | **missing** — no sowing calendar anywhere in the export; the fertilizer file is on a DVS axis, not a date axis | constant DOY 270 |
| `site.altitude` | **missing** — no DEM was exported | 0 m a.s.l. |
| `site.co2` | **missing** | year-dependent global mean (table below) |
| `soil.pmini` / `soil.kmini` | **missing** — SoilGrids carries no P or K, and the `Initial*PConcentration_*` columns in `soil.csv` are copied constants from the Brandenburg reference, not measurements | torchcrop defaults |
| `soil.crairc`, `ksub`, `runfr`, `cfev` | **missing** — not SoilGrids properties | Brandenburg reference constants |
| irrigation amounts | **missing** — management carries only the binary `vIRR` flag, no depths or dates | `vIRR=1` → LINTUL's automatic refill mode |

Because P and K supply is not real data, the default run mode here is
`iopt = 3` (**water + N limited**). Set `iopt = 4` only if you accept the
default P/K pools.

## 1. Configuration

In [ ]:
from __future__ import annotations

import gzip
import re
import warnings
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from torchcrop import (
    CropParameters,
    Lintul5Model,
    SiteParameters,
    SoilParameters,
    WeatherDriver,
)

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("/data01/FDS/muduchuru/Data/SIMPLACE/europe_torchcrop")
WEATHER_DIR = DATA_DIR / "weather"
SOIL_CSV = DATA_DIR / "soil" / "soil.csv"
FERT_CSV = DATA_DIR / "management" / "fertilizer_winter_wheat.csv"

# The fertilizer-composition table is not part of the europe export; it lives
# next to the SIMPLACE reference the exporter was built from.
COMPOSITION_XML = Path(
    "/beegfs/muduchuru/simplace/Brandenburg_1KM_winter_wheat/data/management"
    "/fertilizer_composition.xml"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} on {device}")

In [ ]:
# --- Target grid (must match _work/config_run.yaml: grid:) ----------------- #
GRID_MIN_LON, GRID_MAX_LON = -17.0, 52.0
GRID_MIN_LAT, GRID_MAX_LAT = 34.0, 72.0
GRID_RES = 0.1
N_LON = int(round((GRID_MAX_LON - GRID_MIN_LON) / GRID_RES))   # 690
N_LAT = int(round((GRID_MAX_LAT - GRID_MIN_LAT) / GRID_RES))   # 380

# --- What to simulate ------------------------------------------------------ #
CROP = "wheat"                 # torchcrop CropParameters preset
HARVEST_YEAR = 2020            # winter wheat sown autumn 2019, harvested 2020
SEASON_START = (9, 1)          # (month, day) of the simulation window start,
SEASON_END = (8, 31)           #   in HARVEST_YEAR-1 and HARVEST_YEAR
IOPT = 3                       # 1 potential | 2 water | 3 water+N | 4 water+NPK
N_CELLS = 24                   # cells to draw; None = every cell (70 705!)
CELL_SEED = 0

# --- Soil bucket ----------------------------------------------------------- #
ROOTZONE_M = 1.0               # depth the single LINTUL bucket represents
PROFILE_BOTTOM_M = 2.0         # bottom of soil.csv; ROOTZONE..here = lower zone

# --- Fill values for inputs the export does not carry ---------------------- #
ASSUMPTIONS = {
    "wind_m_s": 2.0,           # only if the export carries no Windspeed
    "sowing_doy": 270,         # 26 Sep, the Brandenburg reference IDPL
    "altitude_m": 0.0,         # no DEM in the export
    "crairc": 0.07,            # critical air content, Brandenburg constant
    "ksub": 100.0,             # max percolation mm/d, Brandenburg constant
    "runfr": 0.0,              # runoff fraction, Brandenburg constant
    "cfev": 2.0,               # soil-evaporation correction, Brandenburg constant
    "cn_ratio": 12.0,          # soil C:N, to turn SoilGrids carbon into organic N
    "mineralisable_frac": 0.003,  # readily mineralisable share of organic N
    "rtnmins": 0.025,          # daily mineralisation fraction, Brandenburg constant
    "nmini_cap_g_m2": 15.0,    # cap on the organic-N pool (peat cells blow up)
    "nminti_scale": 1.0,       # scale on the exported mineral N — see §4.1
}

# Global mean CO2 [ppm] — no CO2 field is exported, so the season year picks one.
CO2_BY_YEAR = pd.Series(
    {1980: 338.8, 1990: 354.4, 2000: 369.6, 2010: 389.9, 2020: 414.2, 2024: 422.7}
)


def co2_for_year(year: int) -> float:
    """Linearly interpolated global-mean CO2 for a calendar year [ppm]."""
    idx = np.arange(CO2_BY_YEAR.index.min(), CO2_BY_YEAR.index.max() + 1)
    return float(CO2_BY_YEAR.reindex(idx).interpolate().reindex([year]).ffill().bfill().iloc[0])


print(f"grid {N_LAT} x {N_LON} = {N_LAT * N_LON} cells; CO2({HARVEST_YEAR}) = "
      f"{co2_for_year(HARVEST_YEAR):.1f} ppm")

### 1.1 Grid ↔ `SimplaceID` ↔ weather file

`data4simplace` assigns `SimplaceID` row-major from the NW corner
(`grid.py:TargetGrid.cell_table`) and names weather files by the **global**
column/row (`exporters/weather_export.py:144`), so every lookup is arithmetic.

In [ ]:
def id_to_rowcol(simplace_id: int | np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """``SimplaceID`` (1-based, row-major from the NW corner) -> (row, col)."""
    zero = np.asarray(simplace_id, dtype=np.int64) - 1
    return zero // N_LON, zero % N_LON


def rowcol_to_lonlat(row: np.ndarray, col: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Grid indices -> cell-centre (lon, lat) in EPSG:4326."""
    lon = GRID_MIN_LON + GRID_RES / 2.0 + np.asarray(col) * GRID_RES
    lat = GRID_MAX_LAT - GRID_RES / 2.0 - np.asarray(row) * GRID_RES
    return lon, lat


def id_to_lonlat(simplace_id) -> tuple[np.ndarray, np.ndarray]:
    """``SimplaceID`` -> cell-centre (lon, lat)."""
    return rowcol_to_lonlat(*id_to_rowcol(simplace_id))


def weather_path(simplace_id: int) -> Path:
    """Path of the gzipped weather file for a ``SimplaceID``."""
    row, col = id_to_rowcol(int(simplace_id))
    return WEATHER_DIR / f"daily_mean_RES1_C{int(col)}R{int(row)}.csv.gz"


_WX_NAME = re.compile(r"daily_mean_RES1_C(\d+)R(\d+)\.csv\.gz$")


def weather_ids() -> np.ndarray:
    """Every ``SimplaceID`` that has a weather file on disk."""
    ids = []
    for name in WEATHER_DIR.iterdir():
        m = _WX_NAME.match(name.name)
        if m:
            col, row = int(m.group(1)), int(m.group(2))
            ids.append(row * N_LON + col + 1)
    return np.sort(np.asarray(ids, dtype=np.int64))


# Round-trip check against a file that exists.
_probe = 60427
_lon, _lat = id_to_lonlat(_probe)
print(f"SimplaceID {_probe} -> lon {_lon:.2f}, lat {_lat:.2f} -> "
      f"{weather_path(_probe).name} exists={weather_path(_probe).exists()}")

### 1.2 Load the tables

**Load-bearing — every later section depends on `soil_df`, `fert_df` and
`runnable`.** Takes ~30 s: `soil.csv` is 70 705 × 130, the fertilizer table is
516 k rows, and `weather_ids()` stats 70 705 directory entries.

`runnable` is the weather ∩ soil ∩ management intersection — the cells that can
be simulated without falling back to a default fertilizer plan.

In [ ]:
SENTINEL = -99.9

soil_df = pd.read_csv(SOIL_CSV).set_index("location")
fert_df = pd.read_csv(FERT_CSV)

wx_ids = weather_ids()
soil_ids = soil_df.index.to_numpy()
fert_ids = fert_df["location"].unique()

runnable = np.intersect1d(np.intersect1d(wx_ids, soil_ids), fert_ids)
print(f"weather cells   : {wx_ids.size:>7,}")
print(f"soil cells      : {soil_ids.size:>7,}")
print(f"management cells: {fert_ids.size:>7,}")
print(f"weather ∩ soil  : {np.intersect1d(wx_ids, soil_ids).size:>7,}")
print(f"all three       : {runnable.size:>7,}   <- runnable with a fertilizer plan")
print(f"soil without a fertilizer plan: {np.setdiff1d(soil_ids, fert_ids).size:,}")

## 2. Data availability audit

Reports, per torchcrop input, whether the European export actually carries it.
Anything marked **missing** is filled from `ASSUMPTIONS` and should be treated
as a model assumption, not as data.

This section is **reporting only** — nothing below depends on it, so it is safe
to skip once you have read it.

In [ ]:
# Column-level audit of one weather file: which columns are real vs all-sentinel.
_probe_wx = pd.read_csv(weather_path(runnable[0]), sep="\t")
wx_report = pd.DataFrame(
    {
        "n_sentinel": (_probe_wx.select_dtypes("number") == SENTINEL).sum(),
        "n_rows": len(_probe_wx),
    }
)
wx_report["status"] = np.where(
    wx_report["n_sentinel"] == wx_report["n_rows"], "MISSING (all -99.9)",
    np.where(wx_report["n_sentinel"] > 0, "partial", "ok"),
)
print(f"{weather_path(runnable[0]).name}  "
      f"{_probe_wx['Date'].iloc[0]} .. {_probe_wx['Date'].iloc[-1]}")
wx_report

In [ ]:
AUDIT = [
    # (torchcrop input, source column / derivation, status)
    ("weather.doy",        "Date",                                   "ok"),
    ("weather.davtmp",     "(TempMin + TempMax) / 2  [SIMPLACE parity]", "ok"),
    ("weather.tmin",       "TempMin",                                "ok"),
    ("weather.tmax",       "TempMax",                                "ok"),
    ("weather.irrad",      "Radiation x 0.0864  [W/m2 -> MJ/m2/d]",   "ok"),
    ("weather.rain",       "Precipitation",                          "ok"),
    ("weather.vp",         "e_s(TempMean) x RelHumCalc/100",         "derived"),
    ("weather.wind",       "Windspeed if exported, else assumed",    "export-dep"),
    ("soil.wcfc",          "soilwater_fc_1..6",                      "ok"),
    ("soil.wcwp",          "soilwater_wp_1..6",                      "ok"),
    ("soil.wcst",          "soilwater_sat_1..6",                     "ok"),
    ("soil.wci/wci_lower", "soilwater_init_1..6",                    "ok"),
    ("soil.wcad",          "soilwater_res_1..6",                     "ok"),
    ("soil.rdmso",         "SoilLayerDepth_6",                       "ok"),
    ("soil.nminti",        "(ammonium + nitrate)_1..6 / 10",         "ok"),
    ("soil.nmini",         "carbon_1..6 / C:N x mineralisable_frac", "assumed"),
    ("soil.crairc",        "not a SoilGrids property",               "MISSING"),
    ("soil.ksub",          "not a SoilGrids property",               "MISSING"),
    ("soil.runfr",         "not a SoilGrids property",               "MISSING"),
    ("soil.cfev",          "not a SoilGrids property",               "MISSING"),
    ("soil.pmini",         "SoilGrids has no P; soil.csv P cols are constants", "MISSING"),
    ("soil.kmini",         "SoilGrids has no K",                     "MISSING"),
    ("site.latitude",      "SimplaceID -> grid",                     "ok"),
    ("site.altitude",      "no DEM exported",                        "MISSING"),
    ("site.co2",           "no CO2 field exported",                  "MISSING"),
    ("site.idpl",          "no sowing calendar exported",            "MISSING"),
    ("fertilizer N/P/K",   "Amount x composition, DVS axis",         "ok"),
    ("irrigation amounts", "management carries only the vIRR flag",  "MISSING"),
]
audit = pd.DataFrame(AUDIT, columns=["torchcrop input", "source", "status"])
audit

## 3. Weather

The export is tab-delimited with a `-99.9` sentinel and columns
`Date, Precipitation, TempMin, TempMean, TempMax, Radiation, Windspeed, RefET,
Gridcell, RelHumCalc`.

Two conversions and one fill:

* **Radiation.** MSWX `rsds` is a daily-mean flux in W m⁻², written through
  unconverted (`weather_export.py:31`). LINTUL wants MJ m⁻² d⁻¹, so
  `× 86400/10⁶ = × 0.0864`. The probe cell peaks at ~331 W m⁻² → 28.6 MJ m⁻² d⁻¹,
  the clear-sky ceiling at 63 °N — the units check out.
  *(This differs from the Brandenburg example, whose `Radiation` is kJ m⁻² d⁻¹
  and is divided by 1000.)*
* **Vapour pressure.** Not exported. Reconstructed from the exported relative
  humidity with Tetens: `e_s(T) = 0.6108 · exp(17.27 T / (T + 237.3))` kPa,
  `vp = e_s(T_mean) · RH/100`.
* **Wind.** `Windspeed` is `-99.9` in every row of the current export, so it
  falls back to `ASSUMPTIONS["wind_m_s"]`. That is an exporter gap, not a data
  gap: MSWX ships `SFCWIND`, and `exporters/weather_export.py` now maps
  `sfcwind → Windspeed`, converting the 10 m product to the 2 m equivalent
  SIMPLACE expects (FAO-56 eq. 47, factor 0.748). Re-export the weather files
  and the loader below picks the real column up with no flag to change — it
  falls back only when the column is entirely sentinel. Wind drives the Penman
  ET₀ aerodynamic term, so until then the water balance carries that constant.

`davtmp` is recomputed as `(TempMin + TempMax)/2` rather than taken from
`TempMean`, matching SIMPLACE's `Phenology.java` and the torchcrop examples.

In [ ]:
WX_COLUMNS = [
    "Date", "Precipitation", "TempMin", "TempMean", "TempMax",
    "Radiation", "Windspeed", "RefET", "Gridcell", "RelHumCalc",
]
W_PER_M2_TO_MJ = 86_400.0 / 1e6      # 0.0864


def saturation_vp(temp_c: np.ndarray) -> np.ndarray:
    """Tetens saturation vapour pressure [kPa] for air temperature [°C]."""
    return 0.6108 * np.exp(17.27 * temp_c / (temp_c + 237.3))


def load_weather(simplace_id: int, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """Daily forcing for one cell in torchcrop channel order, [start, end].

    Returns a frame with columns ``doy, davtmp, tmin, tmax, irrad, rain, vp,
    wind`` — the eight `WeatherDriver` channels, in order.
    """
    raw = pd.read_csv(
        weather_path(simplace_id), sep="\t", parse_dates=["Date"],
        na_values=[SENTINEL, -99.0],
    )
    raw = raw[(raw["Date"] >= start) & (raw["Date"] <= end)].reset_index(drop=True)

    tmin, tmax = raw["TempMin"], raw["TempMax"]
    tmean_obs = raw["TempMean"]
    # Use the exported wind whenever the file actually carries it; an export
    # made before `sfcwind` reached the weather exporter is all -99.9, which
    # read_csv turned into NaN above.
    has_wind = raw["Windspeed"].notna().any()
    out = pd.DataFrame(
        {
            "doy": raw["Date"].dt.dayofyear,
            "davtmp": (tmin + tmax) / 2.0,
            "tmin": tmin,
            "tmax": tmax,
            "irrad": raw["Radiation"] * W_PER_M2_TO_MJ,
            "rain": raw["Precipitation"],
            "vp": saturation_vp(tmean_obs) * raw["RelHumCalc"] / 100.0,
            "wind": raw["Windspeed"] if has_wind else ASSUMPTIONS["wind_m_s"],
        }
    )
    # A handful of sentinel days (e.g. 1979-01-01) survive; interpolate the
    # continuous fields and treat a missing rain day as dry.
    out["rain"] = out["rain"].fillna(0.0)
    out = out.interpolate(limit_direction="both")
    out.attrs["dates"] = raw["Date"]
    return out


_win_start = pd.Timestamp(HARVEST_YEAR - 1, *SEASON_START)
_win_end = pd.Timestamp(HARVEST_YEAR, *SEASON_END)
_demo = load_weather(runnable[0], _win_start, _win_end)
print(f"{_win_start.date()} .. {_win_end.date()}  ->  {len(_demo)} days")
print("wind source:", "exported Windspeed" if _demo["wind"].std() > 0
      else f"assumed constant {ASSUMPTIONS['wind_m_s']} m/s (column is all sentinel)")
_demo.describe().T.round(2)

## 4. Soil

`soil.csv` is the SIMPLACE six-layer profile (bottoms at 0.1, 0.3, 0.5, 0.7,
1.0, 2.0 m). LINTUL-5 is a **two-bucket** model, so each property is collapsed
to a thickness-weighted mean over the rooting zone (`0 … ROOTZONE_M`) for the
upper bucket and over `ROOTZONE_M … PROFILE_BOTTOM_M` for the lower one.

Nitrogen needs care. `soil.csv` carries two distinct things:

* `ammonium_i` + `nitrate_i` in **kg N ha⁻¹** — the initialisation mineral N
  that `data4simplace` derives as `soil.mineral_n_fraction` (1 %) of the
  SoilGrids total-N stock. This is directly available inorganic N →
  `soil_params.nminti` after `kg ha⁻¹ → g m⁻² = ÷ 10`.
* `carbon_i` in **g kg⁻¹** — SoilGrids SOC. LINTUL's `nmini` is a *mineralisable
  organic* pool, which SoilGrids does not measure, so it is estimated as
  `SOC / (C:N) × mineralisable_frac` and capped. On organic (peat) cells this
  estimate is very large, hence `nmini_cap_g_m2`.

`wcad` uses the exported `soilwater_res_i` (residual water content), clipped
below `wcwp` so the plant-available window never inverts.

In [ ]:
LAYER_BOTTOMS_M = np.array([0.1, 0.3, 0.5, 0.7, 1.0, 2.0])
LAYER_TOPS_M = np.concatenate([[0.0], LAYER_BOTTOMS_M[:-1]])
N_LAYERS = LAYER_BOTTOMS_M.size


def overlap_thickness(top_m: float, bottom_m: float) -> np.ndarray:
    """Thickness [m] of each soil layer inside the window ``[top_m, bottom_m]``."""
    return np.clip(
        np.minimum(LAYER_BOTTOMS_M, bottom_m) - np.maximum(LAYER_TOPS_M, top_m),
        0.0, None,
    )


def layer_values(frame: pd.DataFrame, stem: str) -> np.ndarray:
    """``[n_cells, 6]`` array of ``<stem>_1 .. <stem>_6``."""
    return frame[[f"{stem}_{i}" for i in range(1, N_LAYERS + 1)]].to_numpy(float)


def depth_mean(frame: pd.DataFrame, stem: str, top_m: float, bottom_m: float) -> np.ndarray:
    """Thickness-weighted mean of a layered property over a depth window."""
    w = overlap_thickness(top_m, bottom_m)
    return layer_values(frame, stem) @ w / w.sum()


def depth_sum(frame: pd.DataFrame, stem: str, top_m: float, bottom_m: float) -> np.ndarray:
    """Depth-integrated *stock*: per-layer values already are per-layer totals."""
    w = overlap_thickness(top_m, bottom_m) / (LAYER_BOTTOMS_M - LAYER_TOPS_M)
    return layer_values(frame, stem) @ w


def build_soil_params(ids: np.ndarray, dtype=torch.float32) -> SoilParameters:
    """Collapse the 6-layer SIMPLACE profile into LINTUL-5 bucket parameters."""
    f = soil_df.loc[ids]
    root, deep = (0.0, ROOTZONE_M), (ROOTZONE_M, PROFILE_BOTTOM_M)

    wcwp = depth_mean(f, "soilwater_wp", *root)
    wcfc = depth_mean(f, "soilwater_fc", *root)
    wcst = depth_mean(f, "soilwater_sat", *root)
    wcad = np.minimum(depth_mean(f, "soilwater_res", *root), wcwp * 0.999)
    wci = np.clip(depth_mean(f, "soilwater_init", *root), wcwp, wcst)

    # Mineral N (kg N/ha over the rooting zone) -> g N/m2.
    nminti = (
        depth_sum(f, "ammonium", *root) + depth_sum(f, "nitrate", *root)
    ) / 10.0 * ASSUMPTIONS["nminti_scale"]

    # Mineralisable organic N from SOC: g C/kg x kg/dm3 -> g C/dm3;
    # x 100 dm3/m2 per 0.1 m of depth -> g C/m2; then / (C:N).
    carbon = layer_values(f, "carbon")                 # g C / kg soil
    bd = layer_values(f, "bulkdensity")                # kg / dm3
    thick_dm = overlap_thickness(*root) * 10.0         # m -> dm
    c_stock = (carbon * bd * thick_dm * 100.0).sum(axis=1)   # g C / m2
    nmini = np.clip(
        c_stock / ASSUMPTIONS["cn_ratio"] * ASSUMPTIONS["mineralisable_frac"],
        1.0, ASSUMPTIONS["nmini_cap_g_m2"],
    )

    n = len(ids)
    const = lambda v: torch.full((n,), float(v), dtype=dtype)
    t = lambda a: torch.as_tensor(np.asarray(a, float), dtype=dtype)

    return SoilParameters(
        wcad=t(wcad), wcwp=t(wcwp), wcfc=t(wcfc), wcst=t(wcst),
        wci=t(wci), wci_lower=t(np.clip(depth_mean(f, "soilwater_init", *deep),
                                        depth_mean(f, "soilwater_wp", *deep),
                                        depth_mean(f, "soilwater_sat", *deep))),
        crairc=const(ASSUMPTIONS["crairc"]),
        ksub=const(ASSUMPTIONS["ksub"]),
        runfr=const(ASSUMPTIONS["runfr"]),
        cfev=const(ASSUMPTIONS["cfev"]),
        rdmso=const(ROOTZONE_M),
        irri=torch.zeros(n, dtype=dtype),
        rtnmins=const(ASSUMPTIONS["rtnmins"]),
        nmini=t(nmini),
        nminti=t(nminti),
        # pmini / kmini have no European data -> torchcrop defaults.
        pmini=const(SoilParameters().pmini.item()),
        kmini=const(SoilParameters().kmini.item()),
    )


def build_site_params(ids: np.ndarray, year: int, dtype=torch.float32) -> SiteParameters:
    """Latitude from the grid; altitude, CO2 and sowing DOY from ASSUMPTIONS."""
    _, lat = id_to_lonlat(ids)
    n = len(ids)
    const = lambda v: torch.full((n,), float(v), dtype=dtype)
    return SiteParameters(
        latitude=torch.as_tensor(lat, dtype=dtype),
        altitude=const(ASSUMPTIONS["altitude_m"]),
        co2=const(co2_for_year(year)),
        plant_at_sowing=torch.ones(n, dtype=dtype),
        idpl=const(ASSUMPTIONS["sowing_doy"]),
    )

### 4.1 Sanity-check the nitrogen pools

The two N pools decide whether the run is N-limited at all, and both come out of
the export at a questionable magnitude — check them before trusting any
`iopt ≥ 3` result.

* **`nminti`** (initial mineral N) is what `data4simplace` wrote:
  `soil.mineral_n_fraction = 0.01` of the SoilGrids **total** N stock. Over a
  1 m profile that is a domain-wide median of **243 kg N ha⁻¹** (5–95 %:
  130–453, max 1 408 across all 70 705 cells), whereas measured
  pre-sowing N<sub>min</sub> (0–90 cm) in European arable soils is typically
  **30–90 kg N ha⁻¹**. The exported value looks 4–10× too high, which would make
  the crop effectively N-unlimited and flatten the fertilizer response.
  `ASSUMPTIONS["nminti_scale"]` exists to test that sensitivity — it is `1.0`
  here so the cell below reports the export unmodified.
* **`nmini`** (mineralisable organic N) has no direct source. LINTUL releases
  roughly the whole pool over a season (`rtnmins = 0.025 d⁻¹` × ~250 days), so
  the pool *is* the seasonal mineralisation total. The Brandenburg reference
  uses `NMINS = 3 g N m⁻²`; `mineralisable_frac` is set so the SOC-derived
  estimate lands in the same 3–15 g m⁻² band rather than the ~120 g m⁻² a naive
  C:N split gives.

In [ ]:
_probe_soil = build_soil_params(runnable[::50])       # ~1 400 cells across Europe
n_diag = pd.DataFrame(
    {
        "nminti [kg N/ha]": _probe_soil.nminti.numpy() * 10,
        "nmini  [kg N/ha]": _probe_soil.nmini.numpy() * 10,
    }
).describe().T.round(1)
print("typical measured pre-sowing Nmin (0-90 cm): 30-90 kg N/ha\n")
n_diag

## 5. Management

`fertilizer_winter_wheat.csv` is the SIMPLACE long table — one row per cell,
event and product — with `Amount` in **grams of product per m²** placed at a
development stage `DVS`, plus the binary `vIRR` irrigation flag.

torchcrop wants the opposite shape: a `[B, T, 3]` tensor of **elemental N, P, K
in g m⁻² d⁻¹ on calendar days**. Two conversions:

1. **Product → element**, from `fertilizer_composition.xml`: `NitrateAndAmmonium`
   is g N per g of product (KAS = 0.27), `Phosphorus` g P per g (P = 0.4364,
   i.e. already the P₂O₅→P factor), `Potassium` g K per g (K = 0.8302).
2. **DVS → day index.** Nothing on disk maps a DVS to a date, and DVS depends on
   the weather. Rather than guessing, §7 runs the model **once unfertilised** to
   get each cell's own DVS trajectory, then places every dose on the first day
   that reaches its DVS. Phenology in LINTUL-5 is temperature-driven, so the
   unfertilised trajectory is the right schedule for the fertilised run.

`vIRR` is a flag, not a schedule — there are no irrigation depths or dates in
the export. Cells flagged irrigated are given LINTUL's **automatic** mode
(`irri = 1`, refill to field capacity on demand); rainfed cells get `irri = 0`.

In [ ]:
def load_composition(path: Path) -> pd.DataFrame:
    """Fertilizer type -> elemental N, P, K content [g element / g product]."""
    root = ET.parse(path).getroot()
    rows = {}
    for fert in root.findall("fertilizer"):
        params = {p.get("id"): p.text.strip() for p in fert.findall("parameter")}
        rows[params["Fertilizertype"]] = {
            "N": float(params["NitrateAndAmmonium"]),
            "P": float(params["Phosphorus"]),
            "K": float(params["Potassium"]),
        }
    return pd.DataFrame(rows).T


composition = load_composition(COMPOSITION_XML)
composition

In [ ]:
# Long table -> per-cell event list carrying elemental amounts.
fert_events = fert_df.join(composition, on="vType")
missing_types = fert_events.loc[fert_events["N"].isna(), "vType"].unique()
if missing_types.size:
    raise KeyError(f"products absent from the composition file: {missing_types}")

for element in ("N", "P", "K"):
    fert_events[f"{element}_g_m2"] = fert_events["Amount"] * fert_events[element]

fert_events = fert_events[
    ["location", "Event", "vType", "DVS", "Amount", "N_g_m2", "P_g_m2", "K_g_m2", "vIRR"]
]
irrigated_flag = fert_df.groupby("location")["vIRR"].max()

print(f"{fert_events['location'].nunique():,} cells, "
      f"{len(fert_events):,} events; irrigated cells: {int(irrigated_flag.sum()):,}")
fert_events.head(8)

In [ ]:
def fertilizer_from_dvs(
    ids: np.ndarray, dvs: torch.Tensor, dtype=torch.float32
) -> torch.Tensor:
    """Place each cell's DVS-keyed doses onto calendar days -> ``[B, T, 3]``.

    Args:
        ids: ``SimplaceID`` per batch row.
        dvs: ``[B, T+1]`` development-stage trajectory from a first pass.

    Returns:
        Daily N, P, K in g element m^-2 d^-1, last axis ordered (N, P, K).
    """
    dvs_np = dvs.detach().cpu().numpy()[:, 1:]     # drop the pre-sowing state
    n_batch, n_days = dvs_np.shape
    out = np.zeros((n_batch, n_days, 3))
    by_cell = {loc: g for loc, g in fert_events.groupby("location")}

    for b, sid in enumerate(ids):
        events = by_cell.get(int(sid))
        if events is None:                          # no plan -> unfertilised
            continue
        track = dvs_np[b]
        for _, ev in events.iterrows():
            reached = np.flatnonzero(track >= ev["DVS"])
            if reached.size == 0:                   # DVS never reached this season
                continue
            # DVS 0.001 fires on the sowing day itself, once DVS leaves zero.
            day = int(reached[0])
            out[b, day, 0] += ev["N_g_m2"]
            out[b, day, 1] += ev["P_g_m2"]
            out[b, day, 2] += ev["K_g_m2"]
    return torch.as_tensor(out, dtype=dtype)

## 6. Assemble the batch

Every torchcrop batch row is one (cell, season). Cells are drawn from the
weather ∩ soil ∩ management intersection so no row falls back to a default plan.

In [ ]:
rng = np.random.default_rng(CELL_SEED)
cell_ids = runnable if N_CELLS is None else np.sort(
    rng.choice(runnable, size=min(N_CELLS, runnable.size), replace=False)
)

win_start = pd.Timestamp(HARVEST_YEAR - 1, *SEASON_START)
win_end = pd.Timestamp(HARVEST_YEAR, *SEASON_END)

frames = [load_weather(sid, win_start, win_end) for sid in cell_ids]
n_days = min(len(f) for f in frames)
weather_np = np.stack([f.iloc[:n_days].to_numpy(float) for f in frames])  # [B, T, 8]
dates = frames[0].attrs["dates"].iloc[:n_days].reset_index(drop=True)

weather = WeatherDriver(torch.as_tensor(weather_np, dtype=torch.float32))
soil_params = build_soil_params(cell_ids)
site_params = build_site_params(cell_ids, HARVEST_YEAR)

# vIRR -> LINTUL automatic irrigation mode.
soil_params.irri = torch.as_tensor(
    irrigated_flag.reindex(cell_ids).fillna(0).to_numpy(float), dtype=torch.float32
)

crop_params = CropParameters(crop_name=CROP)
crop_params.iopt = torch.tensor(float(IOPT))

start_doy = int(dates.iloc[0].dayofyear)
print(f"batch B={weather.batch_size}, T={weather.n_days} days "
      f"({dates.iloc[0].date()} .. {dates.iloc[-1].date()}), start_doy={start_doy}")
print(f"irrigated cells in batch: {int(soil_params.irri.sum().item())}")

pd.DataFrame(
    {
        "SimplaceID": cell_ids,
        "lon": np.round(id_to_lonlat(cell_ids)[0], 2),
        "lat": np.round(id_to_lonlat(cell_ids)[1], 2),
        "wcwp": np.round(soil_params.wcwp.numpy(), 3),
        "wcfc": np.round(soil_params.wcfc.numpy(), 3),
        "wcst": np.round(soil_params.wcst.numpy(), 3),
        "nminti_gN_m2": np.round(soil_params.nminti.numpy(), 2),
        "nmini_gN_m2": np.round(soil_params.nmini.numpy(), 1),
        "irri": soil_params.irri.numpy().astype(int),
    }
).head(12)

## 7. Run — pass 1 (phenology) then pass 2 (fertilised)

Pass 1 runs unfertilised purely to obtain each cell's DVS trajectory, which
turns the DVS-keyed fertilizer schedule into calendar days. Pass 2 is the run
whose results are reported.

In [ ]:
crop_params = crop_params.to(device=device)
soil_params = soil_params.to(device=device)
site_params = site_params.to(device=device)
weather = weather.to(device=device)

model = Lintul5Model(crop_params, soil_params, site_params).eval().to(device)

with torch.no_grad():
    pass1 = model(weather, start_doy=start_doy)

fertilizer = fertilizer_from_dvs(cell_ids, pass1.dvs).to(device)
applied = fertilizer.sum(dim=1).cpu().numpy()
print(f"pass 1: max DVS {pass1.dvs.max().item():.2f}, "
      f"{int((pass1.dvs[:, -1] >= 2.0).sum())}/{len(cell_ids)} cells reached maturity")
print(f"applied per cell [g/m2]  N {applied[:, 0].mean():.1f}  "
      f"P {applied[:, 1].mean():.2f}  K {applied[:, 2].mean():.2f}")

In [ ]:
with torch.no_grad():
    out = model(weather, start_doy=start_doy, fertilizer=fertilizer)

results = pd.DataFrame(
    {
        "SimplaceID": cell_ids,
        "lon": np.round(id_to_lonlat(cell_ids)[0], 2),
        "lat": np.round(id_to_lonlat(cell_ids)[1], 2),
        "yield_g_m2": np.round(out.yield_.cpu().numpy(), 1),
        "adjusted_yield_g_m2": np.round(out.adjusted_yield.cpu().numpy(), 1),
        "max_lai": np.round(out.lai.max(dim=1).values.cpu().numpy(), 2),
        "final_dvs": np.round(out.dvs[:, -1].cpu().numpy(), 2),
        "N_applied_g_m2": np.round(applied[:, 0], 1),
        "irri": soil_params.irri.cpu().numpy().astype(int),
    }
)
results["yield_t_ha"] = (results["yield_g_m2"] / 100.0).round(2)
results.sort_values("yield_t_ha", ascending=False).head(15)

## 8. Results

In [ ]:
# Okabe-Ito: an 8-hue categorical set that stays separable under all three
# common CVD types. Assigned in fixed order, never cycled.
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#F0E442", "#000000"]
GRID_KW = dict(color="0.9", linewidth=0.8)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})

def state_trajectory(name: str) -> np.ndarray:
    """``[B, T+1]`` trajectory of a `ModelState` field."""
    return torch.stack([getattr(s, name) for s in out.states], dim=1).cpu().numpy()


def diagnostic_trajectory(name: str) -> np.ndarray:
    """``[B, T]`` trajectory of a `DiagnosticState` field."""
    return torch.stack([getattr(d, name) for d in out.diagnostics], dim=1).cpu().numpy()


panels = [
    ("DVS [-]", out.dvs.cpu().numpy()[:, 1:]),
    ("LAI [m² m⁻²]", out.lai.cpu().numpy()[:, 1:]),
    ("Storage organs [g m⁻²]", state_trajectory("wso")[:, 1:]),
    ("Rooted-zone water [mm]", state_trajectory("wa")[:, 1:]),
    ("Water stress TRANRF [-]", diagnostic_trajectory("tranrf")),
    ("N nutrition index NNI [-]", diagnostic_trajectory("nni")),
]
show = np.argsort(-results["yield_g_m2"].to_numpy())[:4]   # 4 highest-yielding
day = np.arange(out.dvs.shape[1] - 1)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.4))
for ax, (label, traj) in zip(axes.ravel(), panels):
    for colour, b in zip(OKABE_ITO, show):
        ax.plot(day, traj[b], color=colour, linewidth=2, label=f"{cell_ids[b]}")
    ax.set_xlabel("day of simulation")
    ax.set_ylabel(label)
    ax.grid(True, **GRID_KW)
    ax.set_axisbelow(True)

axes.ravel()[0].legend(title="SimplaceID", frameon=False, fontsize=8,
                       title_fontsize=8)
fig.suptitle(
    f"LINTUL-5 trajectories — winter wheat, harvest {HARVEST_YEAR}, iopt={IOPT}",
    x=0.02, ha="left", fontsize=11,
)
fig.tight_layout()
plt.show()

In [ ]:


fig, (ax_map, ax_hist) = plt.subplots(1, 2, figsize=(11, 4),
                                      gridspec_kw={"width_ratios": [1.35, 1]})

sc = ax_map.scatter(
    results["lon"], results["lat"], c=results["yield_t_ha"],
    cmap="YlGnBu", s=70, edgecolor="white", linewidth=0.8, vmin=0,
)
ax_map.set_xlabel("longitude [°E]")
ax_map.set_ylabel("latitude [°N]")
ax_map.set_title("Simulated yield by cell", loc="left")
ax_map.grid(True, **GRID_KW)
ax_map.set_axisbelow(True)
fig.colorbar(sc, ax=ax_map, label="yield [t ha⁻¹]", shrink=0.9)

ax_hist.hist(results["yield_t_ha"], bins=12, color=OKABE_ITO[0],
             edgecolor="white", linewidth=1.2)
median = results["yield_t_ha"].median()
ax_hist.axvline(median, color=OKABE_ITO[1], linewidth=2)
ax_hist.annotate(f"median {median:.1f} t ha⁻¹",
                 xy=(median, ax_hist.get_ylim()[1] * 0.92),
                 xytext=(6, 0), textcoords="offset points",
                 color=OKABE_ITO[1], fontsize=9)
ax_hist.set_xlabel("yield [t ha⁻¹]")
ax_hist.set_ylabel("cells")
ax_hist.set_title("Distribution", loc="left")
ax_hist.grid(True, axis="y", **GRID_KW)
ax_hist.set_axisbelow(True)

fig.tight_layout()
plt.show()

results[["yield_t_ha", "max_lai", "final_dvs", "N_applied_g_m2"]].describe().T.round(2)

## 9. Scaling up

The batch above is a sample. To sweep the full domain:

* **Cells.** `N_CELLS = None` gives all 68 685 runnable cells. LINTUL-5 is a
  daily Python loop over `T`, so runtime scales with `T`, not `B` — put the
  whole domain in one batch on GPU and it is one loop of ~365 steps. Memory is
  the constraint: `ModelOutput` keeps every trajectory, so ~68 700 × 365 ×
  (states + rates + diagnostics) in float32 will not fit. Chunk at a few
  thousand cells and keep only the summary columns per chunk.
* **Years.** Wrap §6–7 in a loop over `HARVEST_YEAR`; the weather files cover
  1979–2024, so 45 winter-wheat seasons (1980–2024) are available. Reading
  70 705 gzip files per year is the bottleneck — read each file once and slice
  all seasons out of it rather than re-opening per year.
* **Gradients.** Everything here runs under `torch.no_grad()`. Drop that, make
  the target parameters `nn.Parameter`, and the same batch calibrates against
  observed yields — that is the point of torchcrop over SIMPLACE.

## 10. What to fix in the export

Ranked by how much each one moves the result:

1. **Sowing date (`site.idpl`).** A single constant DOY 270 across 34–72 °N is
   the largest error here — real sowing spans early September in Finland to
   November in Iberia. Either export a sowing calendar (MIRCA-OS carries crop
   calendars per cell and is already a dependency of the irrigation stage) or
   derive one from a temperature rule.
2. **Wind speed — fixed in the exporter, needs a re-export.** `SFCWIND` was
   already in `climate.variables`, but `weather_export.py` dropped it: a
   variable absent from `_CANONICAL_TO_SIMPLACE` never becomes a column
   whatever the handler loads. That map now carries `sfcwind → Windspeed` and
   converts 10 m → 2 m. Re-run the weather export to fill the column; until
   then ET₀ carries the flat 2 m s⁻¹. `RefET` still has no MSWX source.
3. **Initial mineral N.** Not missing, but probably mis-scaled:
   `soil.mineral_n_fraction = 0.01` of *total* SoilGrids N puts a median of
   243 kg N ha⁻¹ in the 0–1 m profile at sowing (see §4.1), against a measured
   N<sub>min</sub> of 30–90 kg ha⁻¹. At that level the crop is N-unlimited and
   `iopt = 3` behaves like `iopt = 2` — the fertilizer schedule barely moves
   the yield in §8. Worth re-deriving against measured N<sub>min</sub>; a
   fraction nearer 0.002 would land in the observed range.
4. **Irrigation depths.** `vIRR` says *whether*, never *how much* or *when*.
   Automatic-refill mode is a reasonable stand-in but reports potential rather
   than actual irrigation demand.
5. **Soil P and K.** No source at all, so `iopt = 4` runs on torchcrop defaults.
   If NPK limitation matters, a P/K supply layer has to enter the pipeline.
6. **Altitude.** Small effect (psychrometric constant only), but a DEM
   aggregated to the same 10 km grid is cheap.
7. **CO₂.** Currently a scalar per season; a year-varying series is trivial and
   matters for multi-decade runs.

Items 1, 2 and 5 are all cell-level scalars — one extra `site.csv` next to
`soil.csv`, with `SimplaceID, latitude, longitude, altitude, sowing_doy`, would
close them together and match the layout the torchcrop examples expect.